# 03. Инфраструктура экспериментов и первые модели

Шаг 3.1. Контекст — `решения.md`: 3.1 (проверка качества), 3.2 (модели), 1.2 (подвыборка).

- Разбиение по времени (вариант А): обучение — до 01.01.2021, валидация — 2021 год, тест — с 01.01.2022. **Тест здесь не трогаем.**
- Модели — на **валидации**, без подбора параметров: Popularity и ALS (RecTools, `ImplicitALSWrapperModel`).
- Код разбиения и метрик — `ml/src/eval/` (`data.py`, `metrics.py`).

In [1]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")  # implicit советует: иначе потоки OpenBLAS мешают своим потокам ALS

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from implicit.cpu.als import AlternatingLeastSquares
from rectools import Columns
from rectools.models import ImplicitALSWrapperModel, PopularModel

sys.path.insert(0, str(Path.cwd().parent / "src"))
from eval.data import RATING_WEIGHTS, describe_split, load_ratings, subsample_users, time_split, to_dataset
from eval.metrics import K, eval_users, evaluate, paired_bootstrap, with_ci

REPORTS = Path.cwd().parent / "reports" / "metrics"
REPORTS.mkdir(parents=True, exist_ok=True)
pd.options.display.float_format = "{:.4f}".format
pd.options.display.width = 200

FRAC = 0.2   # доля случайных пользователей в подвыборке (1.2)
SEED = 42

ratings = load_ratings()
print(f"Всего оценок: {len(ratings):,}")

Всего оценок: 9,164,925


## 1. Разбиение и подвыборка

In [2]:
def fmt(df):
    out = df.copy()
    for c in ["оценок", "пользователей", "книг"]:
        out[c] = out[c].map("{:,}".format)
    for c in ["польз. с историей", "оценок на новые книги"]:
        if c in out:
            out[c] = out[c].map(lambda v: "" if pd.isna(v) else f"{v:.1%}")
    return out

print("Полные данные:")
display(fmt(describe_split(*time_split(ratings))))

sub = subsample_users(ratings, FRAC, seed=SEED)
train, val, test = time_split(sub)
print(f"\nПодвыборка: {FRAC:.0%} случайных пользователей, книги с ≥ 5 оценками в подвыборке")
display(fmt(describe_split(train, val, test)))

users = eval_users(train, val)
val_hit_users = val.loc[val["rating"] >= 4, Columns.User].nunique()
print(f"\nПользователей для проверки на валидации: {len(users):,} "
      f"(есть попадание 4–5 в 2021 году и история до 2021-го; без истории — {val_hit_users - len(users):,}, это холодный старт)")

Полные данные:


,часть,оценок,пользователей,книг,польз. с историей,оценок на новые книги
0,обучение,"8,104,198","729,654","378,508",,
1,валидация,"504,924","163,668","106,825",93.8%,40.8%
2,тест,"555,803","153,708","107,669",95.4%,45.7%



Подвыборка: 20% случайных пользователей, книги с ≥ 5 оценками в подвыборке


,часть,оценок,пользователей,книг,польз. с историей,оценок на новые книги
0,обучение,"1,138,897","143,916","83,639",,
1,валидация,"69,625","26,665","22,365",91.1%,37.3%
2,тест,"72,737","24,637","21,785",92.7%,40.6%



Пользователей для проверки на валидации: 21,350 (есть попадание 4–5 в 2021 году и история до 2021-го; без истории — 2,257, это холодный старт)


**Подвыборка — 20% случайных пользователей** (1.2): в обучении 1,14 млн оценок, 144 тыс. пользователей, **84 тыс. книг** —
в пределах 50–100 тыс. из 1.2. Полный цикл «обучение ALS → рекомендации → метрики» — около 20 секунд
(замер: 10% — 38 тыс. книг, 6 с; 30% — 128 тыс. книг, 38 с; все данные — 4 мин 20 с). Доли «пользователей с историей»
(91–93%) и «оценок на новые книги» (37–41%) близки к полным данным — подвыборка их сохраняет.

## 2. Модели

- **Popularity** — самые популярные книги обучения (по числу пользователей), кроме уже оценённых.
- **ALS** — матричная факторизация `implicit`, параметры по умолчанию (64 фактора, 15 итераций); подбор — позже (Optuna).
  Два варианта: все оценки с весом 1 и **с весами по оценке** (3.1): 1★ — 0,1, 2★ — 0,25, 3★ — 0,5, 4★ — 0,8, 5★ — 1.

In [3]:
def als(seed=SEED):
    return ImplicitALSWrapperModel(AlternatingLeastSquares(factors=64, iterations=15, random_state=seed, num_threads=0))

MODELS = {
    "Popularity": (lambda: PopularModel(popularity="n_users"), True),
    "ALS (без весов)": (als, False),
    "ALS (с весами)": (als, True),
}

results, per_user, timing = [], {}, []
datasets = {w: to_dataset(train, weighted=w) for w in (True, False)}
for name, (make, weighted) in MODELS.items():
    ds = datasets[weighted]
    model = make()
    t0 = time.time(); model.fit(ds); fit_s = time.time() - t0
    t0 = time.time()
    reco = model.recommend(users, ds, k=K, filter_viewed=True, on_unsupported_targets="ignore")
    reco_s = time.time() - t0
    table, pu = evaluate(reco, train, val, users)
    results.append(with_ci(table, pu).assign(модель=name))
    per_user[name] = pu
    timing.append({"модель": name, "обучение, с": round(fit_s, 1), "рекомендации, с": round(reco_s, 1)})
    print(f"{name}: обучение {fit_s:.0f} с, рекомендации для {len(users):,} пользователей {reco_s:.0f} с")

Popularity: обучение 0 с, рекомендации для 21,350 пользователей 1 с


ALS (без весов): обучение 9 с, рекомендации для 21,350 пользователей 7 с


ALS (с весами): обучение 8 с, рекомендации для 21,350 пользователей 6 с


## 3. Результаты на валидации

In [4]:
res = pd.concat(results, ignore_index=True)
cols = ["модель", "разрез", "пользователей", "NDCG@10", "NDCG@10 95% ДИ", "Recall@10", "Precision@10", "Coverage@10", "Novelty@10"]
res = res[cols]
res.to_csv(REPORTS / "baselines_val.csv", index=False, encoding="utf-8")
display(res.style.format({"пользователей": "{:,}", "NDCG@10": "{:.4f}", "Recall@10": "{:.4f}", "Precision@10": "{:.4f}",
                          "Coverage@10": "{:.4f}", "Novelty@10": "{:.2f}"}, na_rep="").hide(axis="index"))
display(pd.DataFrame(timing).style.hide(axis="index"))

модель,разрез,пользователей,NDCG@10,NDCG@10 95% ДИ,Recall@10,Precision@10,Coverage@10,Novelty@10
Popularity,все,"21,350",0.0024,0.0020–0.0028,0.0051,0.0010,0.0002,6.82
Popularity,старые,"15,945",0.0035,0.0030–0.0041,0.0076,0.0013,,
Popularity,новые,"9,396",0.0000,0.0000–0.0000,0.0000,0.0000,,
ALS (без весов),все,"21,350",0.0042,0.0037–0.0047,0.0070,0.0018,0.0106,8.84
ALS (без весов),старые,"15,945",0.0061,0.0054–0.0069,0.0106,0.0024,,
ALS (без весов),новые,"9,396",0.0000,0.0000–0.0000,0.0000,0.0000,,
ALS (с весами),все,"21,350",0.0042,0.0037–0.0048,0.0070,0.0018,0.0095,8.85
ALS (с весами),старые,"15,945",0.0061,0.0053–0.0069,0.0105,0.0024,,
ALS (с весами),новые,"9,396",0.0000,0.0000–0.0000,0.0000,0.0000,,


модель,"обучение, с","рекомендации, с"
Popularity,0.100000,0.500000
ALS (без весов),8.900000,6.800000
ALS (с весами),8.400000,5.800000


In [5]:
# Разница NDCG@10 «все» — парный bootstrap по одним и тем же пользователям
pairs = [("ALS (с весами)", "Popularity"), ("ALS (без весов)", "Popularity"), ("ALS (с весами)", "ALS (без весов)")]
rows = []
for a, b in pairs:
    r = paired_bootstrap(per_user[a]["все"], per_user[b]["все"])
    rows.append({"сравнение": f"{a} − {b}", "разница NDCG@10": r["разница"], "95% ДИ": f"{r['low']:+.4f} … {r['high']:+.4f}",
                 "значимо": "да" if r["low"] > 0 or r["high"] < 0 else "нет"})
pd.DataFrame(rows).style.format({"разница NDCG@10": "{:+.4f}"}).hide(axis="index")

сравнение,разница NDCG@10,95% ДИ,значимо
ALS (с весами) − Popularity,+0.0018,+0.0011 … +0.0024,да
ALS (без весов) − Popularity,+0.0018,+0.0011 … +0.0025,да
ALS (с весами) − ALS (без весов),+0.0000,-0.0003 … +0.0003,нет


In [6]:
# Какую долю «новых» книг модель вообще может порекомендовать: они не встречались в обучении
new_share = (~val.loc[val["rating"] >= 4, Columns.Item].isin(set(train[Columns.Item]))).mean()
print(f"Доля попаданий валидации на новые книги: {new_share:.1%} — их не порекомендует ни Popularity, ни ALS (нет в обучении)")

Доля попаданий валидации на новые книги: 38.0% — их не порекомендует ни Popularity, ни ALS (нет в обучении)


**Вывод.**
- **ALS значимо лучше Popularity:** NDCG@10 0,0042 против 0,0024 (+75%), 95%-интервал разницы не содержит 0. Popularity всем
  рекомендует одни и те же ~10 книг (Coverage 0,02% каталога), ALS — в 50 раз разнообразнее (Coverage ~1%) и новее (Novelty 8,8 против 6,8).
- **Веса оценок (1★ — 0,1 … 5★ — 1) на ALS по умолчанию не влияют** (разница 0,0000, незначимо): при `alpha = 1` уверенность
  1 + alpha·вес почти одинакова для всех оценок. Проверим вместе с подбором `alpha` (Optuna).
- **«Новые» книги — ноль у обеих моделей:** 38% попаданий валидации приходятся на книги, которых нет в обучении, и 44% пользователей
  проверки читали хотя бы одну такую. Это предел для любой коллаборативной модели — нужен content-based (3.2, модели 2–3) и гибрид (3.3).
- **Абсолютные значения низкие** (в среднем ~0,02 попадания на 10 рекомендаций) — это типично для Amazon Books: каталог в 84 тыс. книг,
  у пользователя в среднем 5–10 оценок, проверка по будущему году. На подвыборках разного размера NDCG тоже разный
  (ALS: 10% — 0,0054, 20% — 0,0042, все данные — 0,0034: чем больше каталог, тем труднее попасть), поэтому
  **сравниваем модели только на одной и той же подвыборке**, а итоговые числа для диплома — на полном наборе (тест).
- Параметры ALS по умолчанию; дальше — подбор (factors, regularization, alpha, iterations) на валидации.